<font size = "4">

**Breadth-First Search**

- We will solve the word ladder problem using the **breadth-first search (BFS)** algorithm.

- Given a starting vertex $s$ from a graph $G$, the search proceeds by exploring edges in the graph to find all vertices in $G$ for which a path exists from $s$.

- It finds *all* vertices that are a distance $k$ from $s$ before it finds *any* vertices that are a distance $k+1$.

- In essence, BFS builds a tree where $s$ is the root and it constructs the tree one level at a time.

- To keep track of progress, the algorithm colors each vertex white, gray, or black. 

    - A white vertex is an undiscovered vertex. All vertices are initialized to white.

    - When a vertex is initially discovered, it is colored gray. Gray nodes may be adjacent to unexplored (white) vertices.

    - When BFS has explored all of a vertex's neighbors it is colored black. Once a vertex is colored black, it has no white vertices adjacent to it.

- We will need to add 3 attributes to the `Vertex` class: 

    - `color` - indicating whether it is white, gray, or black. Initialized to "white"

    - `distance` - how many edges must be traversed to get from some starting vertex to the current Vertex. It will be initialized as `np.inf`.

    - `previous` - if there exists a path from the starting Vertex to the current Vertex, this corresponds to the previous Vertex visited along the path. It will be initialized as `None`.

- We'll also include the function building the word ladder graph

In [9]:
import numpy as np

class Vertex:
    def __init__(self, key, val=None):
        self.key = key 
        self.neighbors = {}
        self.val = val
        self.color = "white"
        self.distance = np.inf 
        self.previous = None
    
    def get_neighbor(self, other_key):
        return self.neighbors.get(other_key) # fixed typo
    
    def set_neighbor(self, other_key, weight = 0):
        self.neighbors[other_key] = weight
    
    def __repr__(self):
        return f"Vertex({self.key})"
    
    def __str__(self):
        return str(self.key) + " connected to: " + str([x.key for x in self.neighbors])

    def get_neighbors(self):
        return self.neighbors.keys()

    def get_key(self):
        return self.key

class Graph:
    def __init__(self):
        self.vertices = {}
    
    def set_vertex(self, key):
        self.vertices[key] = Vertex(key)

    def get_vertex(self, key):
        return self.vertices.get(key)

    def __contains__(self, key):
        return key in self.vertices

    def add_edge(self, from_vert, to_vert, weight=0):
        if from_vert not in self.vertices:
            self.set_vertex(from_vert)
        if to_vert not in self.vertices:
            self.set_vertex(to_vert)
        self.vertices[from_vert].set_neighbor(self.vertices[to_vert], weight)

    def get_vertices(self):
        return self.vertices.keys()

    def __iter__(self):
        return iter(self.vertices.values())

def build_word_ladder(filename):
    buckets = {}
    with open(filename, "r", encoding="utf8") as file_in:
        all_words = file_in.readlines()
    # create buckets of words that differ by 1 letter
    for line in all_words:
        word = line.strip()
        for i, _ in enumerate(word):
            bucket = f"{word[:i]}_{word[i + 1 :]}"
            buckets.setdefault(bucket, set()).add(word)

    the_graph = Graph()
    # add edges between different words in the same bucket
    for similar_words in buckets.values():
        for word1 in similar_words:
            for word2 in similar_words - {word1}:
                the_graph.add_edge(word1, word2)
    return the_graph

<font size = "4">

In the cell below, we define the `bfs` function, construct the word ladder graph, and then call the function to perform the Breadth-First Search:

In [8]:
from datasci531 import Queue

def bfs(start):
    start.distance = 0
    start.color = 'gray'
    vert_queue = Queue()
    vert_queue.enqueue(start)
    while vert_queue.size() > 0:
        current = vert_queue.dequeue()
        for neighbor in current.get_neighbors():
            if neighbor.color == "white":
                neighbor.color = "gray"
                neighbor.distance = current.distance + 1
                neighbor.previous = current
                vert_queue.enqueue(neighbor)
        current.color = "black"


G = build_word_ladder("words.txt")
bfs(G.get_vertex("fool"))

<font size = "4">

**Algorithm description**

- BFS begins at the starting vertex `start` and paints it gray, indicating it is currently being explored. The `.distance` value will indicated how far a Vertex is from `start`, so it is changed to zero. `.previous` is left as `None`, since there is no previous Vertex on the "path" from `start` to itself.

- The Vertex `start` is then placed inside a queue.

- The algorithm then iteratively explores a new Vertex at the front of the queue, by iterating over its adjacency list. The Vertex at the front of the queue is dequeued, and its neighbors are examined. For each adjacent Vertex, we check its color. If it is `white`, we do the following:

    1. The new unexplored Vertex is colored gray.

    2. The predecessor of the neighbor is set to the current Vertex (that was just dequeued)

    3. The distance of the neighbor is set to `current.distance + 1`

    4. The neighbor is added to the end of the queue, scheduling it for further exploration (but not until at least all other neighbors of the current Vertex are explored).

- Once all the niehgbors are examined, the current Vertex is colored black, indicating no further exploration is necessary.

<font size = "4">

Recall the graph that we constructed from "words.txt"

<div style="text-align: center;">
  <img src="files/wordgraph.png" alt="Centered image" width = "750">
  <br>
  <br>
  <figcaption>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

<font size = "4">

If we apply `bfs` starting with the "fool" vertex, we begin by iterating over the four adjacent vertices, coloring them gray, changing their `.previous` attribute to point to "fool", and adding them to the queue:

<div style="text-align: center;">
  <img src="files/bfs1.png" alt="Centered image" width = "400">
  <br>
  <br>
  <figcaption>
  Figure: After exploring start Vertex "fool" (level 0)</figcaption>
</div>


<font size = "4">

Next, we dequeue the Vertices in the queue and explore their neighbors who are still colored white. The Vertex "cool" has two neighbors, "fool" and "pool" that are already accounted for. When we explore the "pool" Vertex, we find a single white-colored Vertex "poll" which we color gray and add to the end of the queue.

<div style="text-align: center;">
  <img src="files/bfs1_5.png" alt="Centered image" width = "400">
  <br>
  <br>
  <figcaption>
  Figure: First Vertex on level 1 with a white-colored neighbor</figcaption>
</div>

<br>

After we explore all the vertices on level 1, here is the state of our non-white vertices and the Queue:

<div style="text-align: center;">
  <img src="files/bfs2.png" alt="Centered image" width = "400">
  <br>
  <br>
  <figcaption>
  Figure: After exploring all Vertices on level 1</figcaption>
</div>

<font size = "4">

After exploring Level 5, here are the states of the vertices of the graph and the queue:

<div style="text-align: center;">
  <img src="files/bfs3.png" alt="Centered image" width = "400">
  <br>
  <br>
  <figcaption>
  Figure: After exploring all Vertices on levels 1-5</figcaption>
</div>

<br>

The algorithm ends by dequeing the "sage" Vertex. All of its neighbors are colored black, so we color this final vertex black and the algorithm ends.

<font size = "4">

- We see that the solution of the "fool -> sage" problem requires 6 transformations (at least when working with this limited dictionary).

- In addition, we have solved other problems along the way. We can start from any vertex and follow the predecessor arrows back to "fool" to find the shortest word ladder between the two words.

- Here is a function we can use to print the optimal word ladder.


In [7]:
def traverse(starting_vertex):
    current = starting_vertex
    while current:
        print(current.key)
        current = current.previous

traverse(G.get_vertex("sage"))

sage
sale
pale
pall
poll
pool
fool


<font size = "4">

**Breadth-First Search Analysis**

Consider the BFS algorithm implemented above:

```python
    def bfs(start):
        start.distance = 0
        start.color = 'gray'
        vert_queue = Queue()
        vert_queue.enqueue(start)
        while vert_queue.size() > 0:
            current = vert_queue.dequeue()
            for neighbor in current.get_neighbors():
                if neighbor.color == "white":
                    neighbor.color = "gray"
                    neighbor.distance = current.distance + 1
                    neighbor.previous = current
                    vert_queue.enqueue(neighbor)
            current.color = "black"
```

- The `while` is executed *at most*, once for each vertex in the graph. I.e., up to $|V|$ iterations.

- The nested `for` loop is executed at most twice for each edge in the graph (total!)

- Thus, the computational cost is $\mathcal{O}(|V|) + \mathcal{O}(|E|) = \mathcal{O}(|V| + |E|)$.

- This neglects the cost of *initializing* the graph, however.

- We could shorten the algorithm by keeping track of how many nodes haven't been changed to gray or black, and terminating the algorithm even when the queue is non-empty.